# Rupert Sim: free Colab test of the NEW Rupert setup (v3)
Tests Rupert's human-sized body and the new scoring rules on a free GPU. **Costs nothing.**
This is a short run (100 million steps). It won't produce a finished runner; it shows whether he is
learning in the right direction and gives us the early funny stages.

1. **Runtime > Change runtime type > T4 GPU > Save**
2. **Runtime > Run all**
3. When asked about Google Drive: **Connect to Google Drive > Allow**. Results go to My Drive > rupert > colab_run3.

In [ ]:
# 1. Check we really have a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Connect Google Drive and PROVE it works before training starts
from google.colab import drive
import os, datetime
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/rupert/colab_run3'
os.makedirs(OUT, exist_ok=True)
with open(OUT + '/drive_check.txt', 'w') as f:
    f.write('Drive is working ' + str(datetime.datetime.now()))
assert os.path.exists(OUT + '/drive_check.txt'), 'Google Drive is NOT saving. Stop and tell Claude.'
print('GOOD: results will be saved in Google Drive > My Drive > rupert > colab_run3')
print('Open drive.google.com now and check you can see that folder before leaving this running.')

In [ ]:
# 3. Download the code and install the tested versions
!rm -rf rupert-learning
!git clone https://github.com/kempyyyy4/rupert-learning.git
%cd rupert-learning
!pip install -q "jax[cuda12]==0.9.2" mujoco playground imageio-ffmpeg
!python -m rupert.check_install

In [ ]:
# 4. Train. The first lines say exactly how many steps will run. 'distance_m' should climb.
!python -m rupert.train --steps 100000000 --early-steps 30000000 --early-every 1000000 --main-checkpoints 20 --out {OUT}

In [ ]:
# 5. Test attempts on every 3rd checkpoint: per-attempt stats + Blender replay files
!python -m rupert.evaluate --run {OUT} --every 3 --attempts 8

In [ ]:
# 6. Learning curves
import json, matplotlib.pyplot as plt
log = json.load(open(OUT + '/progress.json'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([l['step'] for l in log], [l.get('distance_m', 0) for l in log], marker='o'); ax[0].set_title('distance per attempt (m)')
ax[1].plot([l['step'] for l in log], [l.get('time_upright_s', 0) for l in log], marker='o'); ax[1].set_title('time before falling (s)')
for a in ax: a.set_xlabel('training steps')
plt.show()

In [ ]:
# 7. Film the first, a middle and the last checkpoint
import os, glob, mediapy
os.environ['MUJOCO_GL'] = 'egl'
ck = sorted(glob.glob(OUT + '/checkpoints/step_*.pkl'))
for c in [ck[0], ck[len(ck) // 4], ck[-1]]:
    !python -m rupert.render --run {OUT} --checkpoint {c} --seconds 10
    mp4 = c.replace('/checkpoints/', '/').replace('.pkl', '.mp4')
    print(os.path.basename(c)); mediapy.show_video(mediapy.read_video(mp4), fps=25)

In [ ]:
# 8. Safety net: also download everything to your PC as a zip (your browser will ask where to save)
!cd /content/drive/MyDrive/rupert && zip -qr /content/colab_run3.zip colab_run3
from google.colab import files
files.download('/content/colab_run3.zip')

## Done
Two copies of the results: Google Drive > rupert > colab_run3, and the zip your browser just downloaded.
Extract the zip into `E:\AI Channel\Rupert Sim\outputs` and tell Claude. Also screenshot the curves and say what the three videos look like.